In [1]:
import pandas as pd

file = "/Users/ritikkourav/Downloads/ibtracs.ALL.list.v04r01.csv"

df = pd.read_csv(file, low_memory=False)

print(df.shape)
print(df.columns)
df.head()

(722922, 174)
Index(['SID', 'SEASON', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME',
       'NATURE', 'LAT', 'LON',
       ...
       'BOM_GUST_PER', 'REUNION_GUST', 'REUNION_GUST_PER', 'USA_SEAHGT',
       'USA_SEARAD_NE', 'USA_SEARAD_SE', 'USA_SEARAD_SW', 'USA_SEARAD_NW',
       'STORM_SPEED', 'STORM_DIR'],
      dtype='object', length=174)


,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,...,BOM_GUST_PER,REUNION_GUST,REUNION_GUST_PER,USA_SEAHGT,USA_SEARAD_NE,USA_SEARAD_SE,USA_SEARAD_SW,USA_SEARAD_NW,STORM_SPEED,STORM_DIR
0,,Year,,,,,,,degrees_north,degrees_east,...,second,kts,second,ft,nmile,nmile,nmile,nmile,kts,degrees
1,1842298N11080,1842,1,NI,BB,UNNAMED,1842-10-25 03:00:00,NR,10.9,80.3,...,,,,,,,,,9,265
2,1842298N11080,1842,1,NI,BB,UNNAMED,1842-10-25 06:00:00,NR,10.9,79.8,...,,,,,,,,,9,265
3,1842298N11080,1842,1,NI,BB,UNNAMED,1842-10-25 09:00:00,NR,10.8,79.4,...,,,,,,,,,9,265
4,1842298N11080,1842,1,NI,BB,UNNAMED,1842-10-25 12:00:00,NR,10.8,78.9,...,,,,,,,,,9,265


In [3]:
import pandas as pd

df = pd.read_csv("ibtracs.ALL.list.v04r01.csv", low_memory=False)

df = df.iloc[1:]   # remove units row
df["LAT"] = pd.to_numeric(df["LAT"], errors="coerce")
df["LON"] = pd.to_numeric(df["LON"], errors="coerce")
df["ISO_TIME"] = pd.to_datetime(df["ISO_TIME"], errors="coerce")

df = df.dropna(subset=["LAT","LON"])

In [5]:
import numpy as np

# Ensure sorted track order
df = df.sort_values(["SID", "ISO_TIME"])

# Compute movement step
df["dlat"] = df.groupby("SID")["LAT"].diff()
df["dlon"] = df.groupby("SID")["LON"].diff()

# Remove unrealistic jumps
df["dlat"] = df["dlat"].clip(-3,3)
df["dlon"] = df["dlon"].clip(-3,3)

# Movement statistics
mean_dlat = df["dlat"].mean()
std_dlat  = df["dlat"].std()

mean_dlon = df["dlon"].mean()
std_dlon  = df["dlon"].std()

print("Movement statistics")
print(mean_dlat, std_dlat)
print(mean_dlon, std_dlon)

# Extract wind distribution
wind = pd.to_numeric(df["USA_WIND"], errors="coerce")
wind = wind.dropna()

wind_mean = wind.mean()
wind_std = wind.std()

print("Wind distribution")
print(wind_mean, wind_std)

Movement statistics
0.09226719032602307 0.3436242290297493
-0.08152507799206898 0.528891657264241
Wind distribution
50.18637655860349 27.5464234129323


In [6]:
import random

synthetic_tracks = []

N_STORMS = 10000
MAX_STEPS = 40

for s in range(N_STORMS):

    # random starting point (tropics)
    lat = random.uniform(-20,20)
    lon = random.uniform(30,150)

    track = []

    for t in range(MAX_STEPS):

        dlat = np.random.normal(mean_dlat, std_dlat)
        dlon = np.random.normal(mean_dlon, std_dlon)

        lat += dlat
        lon += dlon

        wind = np.random.normal(wind_mean, wind_std)

        track.append({
            "storm_id": s,
            "step": t,
            "lat": lat,
            "lon": lon,
            "wind": wind
        })

    synthetic_tracks.extend(track)

syn_df = pd.DataFrame(synthetic_tracks)

print("Synthetic dataset size")
print(syn_df.shape)

print(syn_df.head())

Synthetic dataset size
(400000, 5)
   storm_id  step        lat         lon       wind
0         0     0   8.845744  121.886501  43.826086
1         0     1   9.570656  122.059917  85.239712
2         0     2   9.638478  121.429518  44.117034
3         0     3   9.889387  121.087242  19.616471
4         0     4  10.123674  120.198812  50.632302


In [7]:
india_tracks = syn_df[
    (syn_df["lat"] >= 5) &
    (syn_df["lat"] <= 35) &
    (syn_df["lon"] >= 65) &
    (syn_df["lon"] <= 100)
]

print("Points affecting India:", india_tracks.shape)

# storms affecting India
storm_ids = india_tracks["storm_id"].unique()

india_storms = syn_df[syn_df["storm_id"].isin(storm_ids)]

print("Synthetic storms affecting India:", len(storm_ids))

Points affecting India: (49971, 5)
Synthetic storms affecting India: 1605


In [8]:
from geopy.distance import geodesic

site = (19.07,72.87)

wind_samples = []

for _,row in india_storms.iterrows():

    storm_point = (row["lat"], row["lon"])

    d = geodesic(site,storm_point).km

    if d < 300:   # influence radius

        wind_samples.append(row["wind"])

print("Wind samples near site:",len(wind_samples))

Wind samples near site: 2231


In [9]:
from scipy.stats import genextreme
import numpy as np

wind_samples = np.array(wind_samples)

params = genextreme.fit(wind_samples)

w50  = genextreme.ppf(0.98,*params)
w100 = genextreme.ppf(0.99,*params)

print("50 year wind speed:", w50)
print("100 year wind speed:", w100)

50 year wind speed: 167.4764616737758
100 year wind speed: 167.47646168118771


In [10]:
syn_df.to_csv("synthetic_cyclones_global.csv", index=False)

In [11]:
india_storms.to_csv("synthetic_cyclones_india.csv", index=False)

In [2]:
import os

folder = "/Users/ritikkourav/Downloads/root/data/sample2"

for file in os.listdir(folder):
    print(file)

sea_surface_temperature.nc
mean_sea_level_pressure.nc
u_component_of_wind.nc
.DS_Store
geopotential.nc
100m_u_component_of_wind.nc
total_column_water_vapour.nc
top_net_thermal_radiation.nc
10m_v_component_of_wind.nc
total_precipitation.nc
specific_humidity.nc
2m_temperature.nc
10m_u_component_of_wind.nc
2m_dewpoint_temperature.nc
100m_v_component_of_wind.nc
temperature.nc
v_component_of_wind.nc


In [15]:
from datetime import datetime, timedelta

def get_dates_2018():
    start = datetime(2018, 4, 17)
    end = datetime(2018, 10, 5)

    dates = []
    current = start

    while current <= end:
        # Sunday=6, Monday=0, Wednesday=2, Thursday=3
        if current.weekday() in [6, 0, 2, 3]:
            dates.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=1)

    return dates


dates = get_dates_2018()

print("Total dates:", len(dates))
print(dates[:100])  # first 10

Total dates: 98
['2018-04-18', '2018-04-19', '2018-04-22', '2018-04-23', '2018-04-25', '2018-04-26', '2018-04-29', '2018-04-30', '2018-05-02', '2018-05-03', '2018-05-06', '2018-05-07', '2018-05-09', '2018-05-10', '2018-05-13', '2018-05-14', '2018-05-16', '2018-05-17', '2018-05-20', '2018-05-21', '2018-05-23', '2018-05-24', '2018-05-27', '2018-05-28', '2018-05-30', '2018-05-31', '2018-06-03', '2018-06-04', '2018-06-06', '2018-06-07', '2018-06-10', '2018-06-11', '2018-06-13', '2018-06-14', '2018-06-17', '2018-06-18', '2018-06-20', '2018-06-21', '2018-06-24', '2018-06-25', '2018-06-27', '2018-06-28', '2018-07-01', '2018-07-02', '2018-07-04', '2018-07-05', '2018-07-08', '2018-07-09', '2018-07-11', '2018-07-12', '2018-07-15', '2018-07-16', '2018-07-18', '2018-07-19', '2018-07-22', '2018-07-23', '2018-07-25', '2018-07-26', '2018-07-29', '2018-07-30', '2018-08-01', '2018-08-02', '2018-08-05', '2018-08-06', '2018-08-08', '2018-08-09', '2018-08-12', '2018-08-13', '2018-08-15', '2018-08-16', '20